
# Machine Learning Fundamentals & Categorical Encoding

## Research and Practical Assignment

**Project:** Predictive Maintenance

**Dataset:** AI4I Predictive Maintenance Dataset

### Objectives
- Understand the differences between supervised and unsupervised learning.
- Explore categorical encoding techniques.
- Apply encoding methods to a real-world dataset.
- Compare encoding dimensions and memory usage.
- Validate the encoded data and investigate unseen categories.
- Prevent data leakage during preprocessing.


In [1]:

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

RANDOM_STATE = 42
TEST_SIZE = 0.20

print("Libraries imported successfully.")


Libraries imported successfully.



## 1. Dataset Loading and Initial Exploration

The first step is to load the preprocessed dataset and examine its basic structure.

This includes:
- Dataset dimensions.
- Feature names.
- Data types.
- Missing values.
- Categorical feature cardinality.
- Target variable distribution.


In [2]:
DATA_PATH = Path("cleaned_data.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH.resolve()}"
    )

df = pd.read_csv(DATA_PATH)

print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDataset Preview:")
display(df.head())

print("\nTarget Distribution:")
print(df["machine_failure"].value_counts())


Dataset Shape: (10000, 14)

Column Names:
['udi', 'product_id', 'type', 'air_temperature_k', 'process_temperature_k', 'rotational_speed_rpm', 'torque_nm', 'tool_wear_min', 'machine_failure', 'twf', 'hdf', 'pwf', 'osf', 'rnf']

Data Types:
udi                        int64
product_id                   str
type                         str
air_temperature_k        float64
process_temperature_k    float64
rotational_speed_rpm       int64
torque_nm                float64
tool_wear_min              int64
machine_failure            int64
twf                        int64
hdf                        int64
pwf                        int64
osf                        int64
rnf                        int64
dtype: object

Missing Values:
udi                      0
product_id               0
type                     0
air_temperature_k        0
process_temperature_k    0
rotational_speed_rpm     0
torque_nm                0
tool_wear_min            0
machine_failure          0
twf                      

,udi,product_id,type,air_temperature_k,process_temperature_k,rotational_speed_rpm,torque_nm,tool_wear_min,machine_failure,twf,hdf,pwf,osf,rnf
0,1,M14860,m,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,l,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,l,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,l,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,l,298.2,308.7,1408,40.0,9,0,0,0,0,0,0



Target Distribution:
machine_failure
0    9661
1     339
Name: count, dtype: int64



## 2. Data Analysis and Feature Selection

Before applying categorical encoding, we need to identify the input features and the target variable.

The dataset contains:
- Numerical features describing machine operating conditions.
- One categorical feature: `type`.
- A binary target: `machine_failure`.

Identifier columns such as `udi` and `product_id` are excluded because they identify individual records rather than describe machine behavior.

Failure indicator columns (`twf`, `hdf`, `pwf`, `osf`, and `rnf`) are excluded to avoid potential target leakage because they are directly related to failure events.


In [3]:

categorical_columns = df.select_dtypes(
    include=["object", "category"]
).columns

print("Categorical Columns:")
print(categorical_columns.tolist())

print("\nCategorical Cardinality:")
for col in categorical_columns:
    print(f"{col}: {df[col].nunique()} unique values")

print("\nCategorical Value Counts:")
for col in categorical_columns:
    display(df[col].value_counts().to_frame("Count"))


# Define features and target

FEATURES = [
    "type",
    "air_temperature_k",
    "process_temperature_k",
    "rotational_speed_rpm",
    "torque_nm",
    "tool_wear_min"
]

TARGET = "machine_failure"

X = df[FEATURES].copy()
y = df[TARGET].copy()

print("\nSelected Features:")
print(X.columns.tolist())

print("\nTarget Variable:")
print(TARGET)

print("\nFeatures Shape:", X.shape)
print("Target Shape:", y.shape)

print("\nTarget Distribution:")
display(y.value_counts().to_frame("Count"))


Categorical Columns:
['product_id', 'type']

Categorical Cardinality:
product_id: 10000 unique values
type: 3 unique values

Categorical Value Counts:


C:\Users\makhl\AppData\Local\Temp\ipykernel_14056\339954108.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(


,Count
product_id,
M14860,1
L47181,1
L47182,1
L47183,1
L47184,1
...,...
M24855,1
H39410,1
M24857,1


,Count
type,
l,6000
m,2997
h,1003



Selected Features:
['type', 'air_temperature_k', 'process_temperature_k', 'rotational_speed_rpm', 'torque_nm', 'tool_wear_min']

Target Variable:
machine_failure

Features Shape: (10000, 6)
Target Shape: (10000,)

Target Distribution:


,Count
machine_failure,
0,9661
1,339



## 3. Train-Test Split

The dataset is divided into:
- 80% training data.
- 20% testing data.

Stratified splitting is used to preserve the original target class distribution in both subsets.

The split is performed before fitting any data-dependent encoder to prevent data leakage.


In [4]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training Features Shape:", X_train.shape)
print("Testing Features Shape:", X_test.shape)

print("\nTraining Target Distribution:")
display(y_train.value_counts(normalize=True).to_frame("Proportion"))

print("\nTesting Target Distribution:")
display(y_test.value_counts(normalize=True).to_frame("Proportion"))


Training Features Shape: (8000, 6)
Testing Features Shape: (2000, 6)

Training Target Distribution:


,Proportion
machine_failure,
0,0.966125
1,0.033875



Testing Target Distribution:


,Proportion
machine_failure,
0,0.966
1,0.034



## 4. Original Data Before Encoding

Before encoding, we inspect the original categorical values and numerical features.

This provides a baseline for comparing the dataset before and after encoding.


In [5]:

print("Original Training Data:")
display(X_train.head())

print("\nOriginal Testing Data:")
display(X_test.head())

print("\nOriginal Number of Features:", X_train.shape[1])


Original Training Data:


,type,air_temperature_k,process_temperature_k,rotational_speed_rpm,torque_nm,tool_wear_min
4058,m,302.0,310.9,1456,47.2,54
1221,m,297.0,308.3,1399,46.4,132
6895,m,301.0,311.6,1357,45.6,137
9863,l,298.9,309.8,1411,56.3,84
8711,l,297.1,308.5,1733,28.7,50



Original Testing Data:


,type,air_temperature_k,process_temperature_k,rotational_speed_rpm,torque_nm,tool_wear_min
2997,l,300.5,309.8,1345,62.7,153
4871,l,303.7,312.4,1513,40.1,135
3858,l,302.5,311.4,1559,37.6,209
951,h,295.6,306.3,1509,35.8,60
6463,h,300.5,310.0,1358,60.4,102



Original Number of Features: 6



## 5. One-Hot Encoding

One-Hot Encoding converts each category into a separate binary column.

Each category is represented by:
- 1 if the observation belongs to that category.
- 0 otherwise.

In this dataset, the `type` feature contains three nominal categories: `h`, `l`, and `m`.

One-Hot Encoding is suitable because these categories do not have a confirmed natural order.

We use `fit_transform()` on the training data and `transform()` on the testing data to prevent data leakage.

The `handle_unknown="ignore"` parameter allows the encoder to handle previously unseen categories.


In [6]:
onehot_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)


X_train_type_onehot = onehot_encoder.fit_transform(
    X_train[["type"]]
)


X_test_type_onehot = onehot_encoder.transform(
    X_test[["type"]]
)


onehot_columns = onehot_encoder.get_feature_names_out(
    ["type"]
)


X_train_onehot = pd.concat(
    [
        X_train.drop(columns=["type"]),
        pd.DataFrame(
            X_train_type_onehot,
            columns=onehot_columns,
            index=X_train.index
        )
    ],
    axis=1
)

X_test_onehot = pd.concat(
    [
        X_test.drop(columns=["type"]),
        pd.DataFrame(
            X_test_type_onehot,
            columns=onehot_columns,
            index=X_test.index
        )
    ],
    axis=1
)


print("Detected Categories:")
print(onehot_encoder.categories_)

print("\nGenerated Columns:")
print(onehot_columns.tolist())

print("\nTraining Data After One-Hot Encoding:")
display(X_train_onehot.head())

print("\nTesting Data After One-Hot Encoding:")
display(X_test_onehot.head())

print("\nOriginal Feature Count:", X_train.shape[1])
print("Encoded Feature Count:", X_train_onehot.shape[1])

print("\nTraining Shape:", X_train_onehot.shape)
print("Testing Shape:", X_test_onehot.shape)


Detected Categories:
[array(['h', 'l', 'm'], dtype=object)]

Generated Columns:
['type_h', 'type_l', 'type_m']

Training Data After One-Hot Encoding:


,air_temperature_k,process_temperature_k,rotational_speed_rpm,torque_nm,tool_wear_min,type_h,type_l,type_m
4058,302.0,310.9,1456,47.2,54,0.0,0.0,1.0
1221,297.0,308.3,1399,46.4,132,0.0,0.0,1.0
6895,301.0,311.6,1357,45.6,137,0.0,0.0,1.0
9863,298.9,309.8,1411,56.3,84,0.0,1.0,0.0
8711,297.1,308.5,1733,28.7,50,0.0,1.0,0.0



Testing Data After One-Hot Encoding:


,air_temperature_k,process_temperature_k,rotational_speed_rpm,torque_nm,tool_wear_min,type_h,type_l,type_m
2997,300.5,309.8,1345,62.7,153,0.0,1.0,0.0
4871,303.7,312.4,1513,40.1,135,0.0,1.0,0.0
3858,302.5,311.4,1559,37.6,209,0.0,1.0,0.0
951,295.6,306.3,1509,35.8,60,1.0,0.0,0.0
6463,300.5,310.0,1358,60.4,102,1.0,0.0,0.0



Original Feature Count: 6
Encoded Feature Count: 8

Training Shape: (8000, 8)
Testing Shape: (2000, 8)



## 6. Frequency Encoding

Frequency Encoding replaces each category with its relative frequency in the training dataset.

The frequency is calculated as:

Frequency(category) = Count(category) / Total training observations

This method represents how common each category is.

Unlike One-Hot Encoding, Frequency Encoding does not increase the number of columns.

The frequency mapping is calculated using training data only and then applied to the testing data.


In [7]:


frequency_map = X_train["type"].value_counts(normalize=True)


X_train_frequency = X_train.copy()
X_test_frequency = X_test.copy()


X_train_frequency["type"] = (
    X_train_frequency["type"].map(frequency_map)
)

X_test_frequency["type"] = (
    X_test_frequency["type"].map(frequency_map).fillna(0)
)


print("Frequency Mapping:")
display(frequency_map.to_frame("Frequency"))

print("\nTraining Data After Frequency Encoding:")
display(X_train_frequency.head())

print("\nTesting Data After Frequency Encoding:")
display(X_test_frequency.head())

print("\nTraining Shape:", X_train_frequency.shape)
print("Testing Shape:", X_test_frequency.shape)


Frequency Mapping:


,Frequency
type,
l,0.603750
m,0.297625
h,0.098625



Training Data After Frequency Encoding:


,type,air_temperature_k,process_temperature_k,rotational_speed_rpm,torque_nm,tool_wear_min
4058,0.297625,302.0,310.9,1456,47.2,54
1221,0.297625,297.0,308.3,1399,46.4,132
6895,0.297625,301.0,311.6,1357,45.6,137
9863,0.603750,298.9,309.8,1411,56.3,84
8711,0.603750,297.1,308.5,1733,28.7,50



Testing Data After Frequency Encoding:


,type,air_temperature_k,process_temperature_k,rotational_speed_rpm,torque_nm,tool_wear_min
2997,0.603750,300.5,309.8,1345,62.7,153
4871,0.603750,303.7,312.4,1513,40.1,135
3858,0.603750,302.5,311.4,1559,37.6,209
951,0.098625,295.6,306.3,1509,35.8,60
6463,0.098625,300.5,310.0,1358,60.4,102



Training Shape: (8000, 6)
Testing Shape: (2000, 6)



## 7. Encoding Methods Comparison

We compare One-Hot Encoding and Frequency Encoding based on:

- Number of generated features.
- Memory consumption.
- Representation of categorical information.

This comparison helps illustrate the trade-offs between the two encoding methods.


In [8]:

def memory_usage_mb(data):
    return data.memory_usage(deep=True).sum() / (1024 ** 2)


comparison = pd.DataFrame({
    "Encoding": ["One-Hot", "Frequency"],
    "Training Features": [
        X_train_onehot.shape[1],
        X_train_frequency.shape[1]
    ],
    "Testing Features": [
        X_test_onehot.shape[1],
        X_test_frequency.shape[1]
    ],
    "Training Memory (MB)": [
        round(memory_usage_mb(X_train_onehot), 4),
        round(memory_usage_mb(X_train_frequency), 4)
    ]
})

display(comparison)


,Encoding,Training Features,Testing Features,Training Memory (MB)
0,One-Hot,8,8,0.5493
1,Frequency,6,6,0.4272



## 8. Encoding Validation

After encoding, we validate the resulting datasets to ensure that:

- The original number of rows is preserved.
- Training and testing datasets have matching columns.
- All encoded features are numeric.
- No missing or infinite values are introduced.


In [9]:

def validate_encoding(train_data, test_data, method_name):

    print(f"\n========== {method_name} VALIDATION ==========")

    print(
        "Training rows preserved:",
        len(train_data) == len(X_train)
    )

    print(
        "Testing rows preserved:",
        len(test_data) == len(X_test)
    )

    print(
        "Same columns:",
        train_data.columns.equals(test_data.columns)
    )

    print(
        "All columns numeric:",
        all(
            pd.api.types.is_numeric_dtype(dtype)
            for dtype in train_data.dtypes
        )
    )

    print(
        "Training missing values:",
        train_data.isna().sum().sum()
    )

    print(
        "Testing missing values:",
        test_data.isna().sum().sum()
    )

    print(
        "Training values finite:",
        np.isfinite(train_data.to_numpy(dtype=float)).all()
    )

    print(
        "Testing values finite:",
        np.isfinite(test_data.to_numpy(dtype=float)).all()
    )


validate_encoding(
    X_train_onehot,
    X_test_onehot,
    "ONE-HOT"
)

validate_encoding(
    X_train_frequency,
    X_test_frequency,
    "FREQUENCY"
)



========== ONE-HOT VALIDATION ==========
Training rows preserved: True
Testing rows preserved: True
Same columns: True
All columns numeric: True
Training missing values: 0
Testing missing values: 0
Training values finite: True
Testing values finite: True

========== FREQUENCY VALIDATION ==========
Training rows preserved: True
Testing rows preserved: True
Same columns: True
All columns numeric: True
Training missing values: 0
Testing missing values: 0
Training values finite: True
Testing values finite: True



## 9. Handling Unseen Categories

An unseen category is a category that appears in new data but was not present during training.

We simulate an unseen category called `unknown` to examine how each encoding method handles it.

One-Hot Encoding uses `handle_unknown="ignore"` to produce an all-zero representation.

Frequency Encoding assigns zero using `fillna(0)` when a category is not found in the training frequency mapping.


In [10]:

unseen_category = pd.DataFrame({
    "type": ["unknown"]
})

onehot_unseen = onehot_encoder.transform(
    unseen_category
)

frequency_unseen = (
    unseen_category["type"]
    .map(frequency_map)
    .fillna(0)
)

print("One-Hot result:")
print(onehot_unseen)

print("\nFrequency result:")
print(frequency_unseen.tolist())


One-Hot result:
[[0. 0. 0.]]

Frequency result:
[0.0]



## 10. Saving Encoded Datasets

The encoded training and testing datasets are saved separately.

This allows the results to be inspected and reused without repeating the encoding process.


In [11]:

OUTPUT_DIR = Path("encoding_results")
OUTPUT_DIR.mkdir(exist_ok=True)

pd.concat(
    [X_train_onehot, y_train.rename(TARGET)],
    axis=1
).to_csv(OUTPUT_DIR / "train_onehot.csv", index=False)

pd.concat(
    [X_test_onehot, y_test.rename(TARGET)],
    axis=1
).to_csv(OUTPUT_DIR / "test_onehot.csv", index=False)

pd.concat(
    [X_train_frequency, y_train.rename(TARGET)],
    axis=1
).to_csv(OUTPUT_DIR / "train_frequency.csv", index=False)

pd.concat(
    [X_test_frequency, y_test.rename(TARGET)],
    axis=1
).to_csv(OUTPUT_DIR / "test_frequency.csv", index=False)

comparison.to_csv(
    OUTPUT_DIR / "encoding_comparison.csv",
    index=False
)

print("All encoded datasets and comparison results saved successfully.")


All encoded datasets and comparison results saved successfully.
